# Réseau piéton

In [1]:
# Geneva Cycle & Pedestrian Network Analysis
# -------------------------------------------------
# This script downloads Open data for the Canton of Geneva. 
# Only run section 5. to dowload data and save the network into segments
# -------------------------------------------------

import osmnx as ox
import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
from shapely.geometry import LineString
from shapely.ops import nearest_points
from shapely.ops import unary_union
from shapely.geometry import Point
import networkx as nx
from shapely.ops import substring


## Segmentation du réseau et export du fichier

Geometrie = ligne

In [2]:
#import network
print("Loading pedestrian segments, replace file path -->")
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/shp_geneva_pedestrian_edges_all'
pedestrian_network_ME = gpd.read_file(f"{file_path}/geneva_pedestrian_edges_all.shp")
pedestrian_network_ME = pedestrian_network_ME.to_crs(epsg=2056)

In [4]:

def split_linestring(geom, segment_length):
    """Split a LineString into segments of a given length."""
    if geom.length <= segment_length:
        return [geom]
    
    segments = []
    start = 0.0
    while start < geom.length:
        end = min(start + segment_length, geom.length)
        seg = substring(geom, start, end)
        segments.append(seg)
        start = end
    return segments

# Desired segment length in meters
segment_length = 50

# Store the results
split_rows = []

print("Splitting LineStrings into fixed-length segments...")
# replace name of the GeoDataFrame with the network you want to split
gdf_to_split = pedestrian_network_ME

for idx, row in gdf_to_split.iterrows():
    geom = row.geometry
    if geom.is_empty or not isinstance(geom, LineString):
        continue
    
    segments = split_linestring(geom, segment_length)
    for seg in segments:
        new_row = row.copy()
        new_row.geometry = seg
        new_row["length"] = seg.length
        split_rows.append(new_row)

# New GeoDataFrame
pedestrian_segments = gpd.GeoDataFrame(split_rows, crs=pedestrian_network_ME.crs)

# Add a unique segment ID
print("Assigning unique segment IDs...")
pedestrian_segments.reset_index(drop=True, inplace=True)
pedestrian_segments["segment_id"] = pedestrian_segments.index.astype(str).str.zfill(6)  # e.g., '000001'

In [5]:
#save to gpkg file
output_file = "step1_pedestrian_segments.gpkg"
pedestrian_segments.to_file(output_file, driver="GPKG")